COLLECT_LIST
--group_concat in SQL

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
data=[
('user1','book1'),
('user1','book2'),
('user2','book2'),
('user2','book4'),
('user3','book1')
]

schema='user string, book string'

df_book=spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy("user").agg(collect_list("book")).display()

###PIVOT

In [0]:
df_sales=spark.read.csv("/Volumes/workspace/raw_gcd/contracts_file/BigMart Sales.csv", header=True, inferSchema=True)
df_sales.display()

In [0]:
df_sales.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

###WHEN-OTHERWISE

######Scenario - 1

In [0]:
df_sales=df_sales.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non-Veg').otherwise('Veg'))
df_sales.display()

In [0]:
df_sales.withColumn('year',when(col('Outlet_Establishment_Year')>'2000','New').otherwise('Old')).display()

In [0]:
df_sales.withColumn('veg_exp_flag',when((col('veg_flag')=='Veg') & (col('Item_MRP')>100), 'Veg_Expensive')\
                                .when((col('veg_flag')=='Veg') & (col('Item_MRP')<100), 'Veg_InExpensive').otherwise('Non-Veg')).display() 

###JOINS 

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

#####inner

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "inner" ).display()

#####left join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "left" ).display()

######right join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "right" ).display()

#####anti join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'], "anti" ).display()

In [0]:
df2.join(df1, df1['dept_id']==df2['dept_id'], "anti" ).display()

##WINDOW FUNCTIONS

####row_number()

In [0]:
df_sales.display() 

In [0]:
from pyspark.sql.window import Window 

In [0]:
df_sales.withColumn('row_num', row_number().over(Window.partitionBy('Item_Fat_Content','Item_Type').orderBy('Item_Identifier'))).display()

###RANK VS DENSE_RANK

In [0]:
df_sales.withColumn('rank', rank().over(Window.orderBy(col('Item_Identifier'))))\
    .withColumn('dense_rank', dense_rank().over(Window.orderBy('Item_Identifier'))).display()

###Cumulative-Sum

In [0]:
df_sales.withColumn('cum_sum',sum('Item_MRP').over(Window.partitionBy('Outlet_Size').orderBy('Item_Type'))).display()

In [0]:
df_sales.withColumn('cum_sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df_sales.agg(sum('Item_MRP')).display()

In [0]:
df_sales.withColumn('total_sum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

#####   USER DEFINED FUNCTIONS (UDF)

In [0]:
def my_func(x) :
  return x*x

In [0]:
my_udf=udf(my_func)

In [0]:
df_sales.withColumn('amt_sq', my_udf(col('Item_MRP'))).display()

###SPARK SQL

In [0]:
df_sales.createTempView('my_view')

In [0]:
%sql
select * from my_view where Item_Type='Dairy' and Item_MRP < 50 and Item_Fat_Content='Low Fat' and Outlet_Location_Type='Tier 2'

In [0]:
df_sql=spark.sql("select * from my_view where Item_Type='Dairy' and Item_MRP < 50 and Item_Fat_Content='Low Fat' and Outlet_Location_Type='Tier 2' ").display()